
# Tableau de bord investisseur immobilier – Notebook interactif

Ce notebook propose des widgets pour explorer des données de locations meublées de courte durée (Airbnb-like) sur plusieurs zones.
Objectif: aider un investisseur à identifier des zones et types de biens intéressants selon ses critères.


In [ ]:

import os
import glob
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except Exception as e:
    print("ipywidgets indisponible. Installez-le si nécessaire: pip install ipywidgets")
    raise

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

DATA_DIR = "."
OUTPUT_DIR = "./cleaned_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:

def to_float_price(series):
    cleaned = series.astype(str).str.replace(r"[^\d\.,-]", "", regex=True)\
                                .str.replace(",", ".", regex=False)\
                                .str.replace(r"(?<=\d)\.(?=\d{3}(\D|$))", "", regex=True)
    return pd.to_numeric(cleaned, errors="coerce")

def parse_date_safe(s):
    return pd.to_datetime(s, errors="coerce", infer_datetime_format=True)

def clean_listings(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if 'id' in df.columns:
        df = df.drop_duplicates(subset=['id'])
    else:
        df = df.drop_duplicates()
    if 'price' in df.columns:
        df['price'] = to_float_price(df['price'])
    for col in ['minimum_nights','number_of_reviews','calculated_host_listings_count','availability_365','number_of_reviews_ltm']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if 'last_review' in df.columns:
        df['last_review'] = parse_date_safe(df['last_review'])
    for c in ['neighbourhood','neighbourhood_group','neighbourhood_cleansed','room_type','name','host_name']:
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip()
    return df

def build_review_stats(reviews: pd.DataFrame) -> pd.DataFrame:
    r = reviews.copy()
    if 'listing_id' not in r.columns and 'listing' in r.columns:
        r = r.rename(columns={'listing':'listing_id'})
    if 'date' in r.columns:
        r['date'] = parse_date_safe(r['date'])
    if 'listing_id' in r.columns:
        if 'id' in r.columns:
            agg = r.groupby('listing_id', as_index=False).agg(review_count=('id','count'),
                                                              last_review=('date','max'))
        else:
            agg = r.groupby('listing_id', as_index=False).agg(review_count=('listing_id','count'),
                                                              last_review=('date','max'))
    else:
        agg = pd.DataFrame(columns=['listing_id','review_count','last_review'])
    return agg

def safe_merge_listings_neigh(listings: pd.DataFrame, neigh: pd.DataFrame) -> pd.DataFrame:
    if neigh is None or neigh.empty:
        return listings.copy()
    cand = [('neighbourhood','neighbourhood'),
            ('neighbourhood','neighbourhood_cleansed'),
            ('neighbourhood_group','neighbourhood_group')]
    l = listings.copy()
    n = neigh.copy()
    for df in (l, n):
        for c in df.columns:
            if df[c].dtype == 'object':
                df[c] = df[c].astype(str).str.strip().str.lower()
    for lk, rk in cand:
        if lk in l.columns and rk in n.columns:
            merged = l.merge(n, left_on=lk, right_on=rk, how='left', suffixes=('','_neigh'))
            return merged
    return listings.copy()

def load_city(prefix: str, base_dir=DATA_DIR):
    lp = os.path.join(base_dir, f"{prefix}_listings.csv")
    rp = os.path.join(base_dir, f"{prefix}_reviews.csv")
    npth = os.path.join(base_dir, f"{prefix}_neighbourhoods.csv")
    listings = pd.read_csv(lp) if os.path.exists(lp) else None
    reviews = pd.read_csv(rp) if os.path.exists(rp) else None
    neigh = pd.read_csv(npth) if os.path.exists(npth) else None
    if listings is None:
        raise FileNotFoundError(f"Fichier introuvable: {lp}")
    listings = clean_listings(listings)
    merged = safe_merge_listings_neigh(listings, neigh if neigh is not None else pd.DataFrame())
    if reviews is not None and not reviews.empty:
        stats = build_review_stats(reviews)
        merged = merged.merge(stats, left_on='id', right_on='listing_id', how='left')
        merged = merged.drop(columns=['listing_id'], errors='ignore')
    return merged

# Discover available cities from files present
def discover_cities(base_dir=DATA_DIR):
    patterns = ["*_listings.csv"]
    cities = set()
    for pat in patterns:
        for p in glob.glob(os.path.join(base_dir, pat)):
            name = os.path.basename(p)
            if name.endswith("_listings.csv"):
                cities.add(name.replace("_listings.csv",""))
    return sorted(list(cities))

cities_available = discover_cities(DATA_DIR)
print("Villes détectées:", cities_available)


In [ ]:

dfs = []
for city in cities_available:
    try:
        dfc = load_city(city, DATA_DIR)
        dfc['city'] = city
        dfs.append(dfc)
        out_path = os.path.join(OUTPUT_DIR, f"cleaned_{city}.csv")
        dfc.to_csv(out_path, index=False)
    except Exception as e:
        print(f"Erreur chargement {city}: {e}")

if dfs:
    data = pd.concat(dfs, ignore_index=True, sort=False)
    combined_path = os.path.join(OUTPUT_DIR, "cleaned_all_cities.csv")
    data.to_csv(combined_path, index=False)
else:
    data = pd.DataFrame()
print("Shape data combinée:", data.shape)
display(data.head())


Widgets de pilotage de l'exploration

In [ ]:

if data.empty:
    print("Aucune donnée disponible. Vérifiez les fichiers en entrée.")
else:
    # Build widget controls
    city_opts = sorted(data['city'].dropna().unique().tolist()) if 'city' in data.columns else []
    room_types = sorted(data['room_type'].dropna().unique().tolist()) if 'room_type' in data.columns else []
    neighs = sorted(data['neighbourhood'].dropna().unique().tolist()) if 'neighbourhood' in data.columns else []

    w_city = widgets.SelectMultiple(options=city_opts, value=tuple(city_opts), description="Villes")
    price_min = np.nanpercentile(data['price'], 1) if 'price' in data.columns and data['price'].notna().any() else 0
    price_max = np.nanpercentile(data['price'], 99) if 'price' in data.columns and data['price'].notna().any() else 1000
    w_price = widgets.FloatRangeSlider(value=[float(price_min), float(price_max)], min=0.0, max=float(price_max*1.2 if price_max else 1000), step=1.0, description="Prix")

    w_room = widgets.SelectMultiple(options=room_types, value=tuple(room_types), description="Type")
    w_min_nights = widgets.IntRangeSlider(value=[1, int(data['minimum_nights'].max()) if 'minimum_nights' in data.columns else 30],
                                          min=0, max=int(data['minimum_nights'].max()) if 'minimum_nights' in data.columns else 365, step=1, description="Nuits min")
    w_min_reviews = widgets.IntSlider(value=0, min=0, max=int(data['number_of_reviews'].max()) if 'number_of_reviews' in data.columns else 100, step=1, description="Min avis")
    w_last_year = widgets.IntRangeSlider(value=[int(data['last_review'].dropna().dt.year.min()) if 'last_review' in data.columns and data['last_review'].notna().any() else 2015,
                                                int(data['last_review'].dropna().dt.year.max()) if 'last_review' in data.columns and data['last_review'].notna().any() else datetime.now().year],
                                         min=2010, max=datetime.now().year, step=1, description="Années avis")
    w_outliers = widgets.Checkbox(value=True, description="Exclure outliers prix (1-99 pct)")
    w_neigh_filter = widgets.Text(value="", description="Quartier contient")
    w_topn = widgets.IntSlider(value=10, min=5, max=50, step=1, description="Top N quartiers")
    w_bins = widgets.IntSlider(value=30, min=5, max=100, step=1, description="Bins histogramme")
    
    ui = widgets.VBox([w_city, w_room, w_price, w_min_nights, w_min_reviews, w_last_year, w_outliers, w_neigh_filter, w_topn, w_bins])
    display(ui)


In [ ]:

def apply_filters(df):
    d = df.copy()
    if w_outliers.value and 'price' in d.columns:
        lo, hi = np.nanpercentile(d['price'], 1), np.nanpercentile(d['price'], 99)
        d = d[(d['price']>=lo) & (d['price']<=hi)]
    if 'city' in d.columns and len(w_city.value)>0:
        d = d[d['city'].isin(list(w_city.value))]
    if 'room_type' in d.columns and len(w_room.value)>0:
        d = d[d['room_type'].isin(list(w_room.value))]
    if 'price' in d.columns:
        lo, hi = w_price.value
        d = d[(d['price']>=lo) & (d['price']<=hi)]
    if 'minimum_nights' in d.columns:
        lo, hi = w_min_nights.value
        d = d[(d['minimum_nights']>=lo) & (d['minimum_nights']<=hi)]
    if 'number_of_reviews' in d.columns:
        d = d[d['number_of_reviews']>=w_min_reviews.value]
    if 'last_review' in d.columns and d['last_review'].notna().any():
        ylo, yhi = w_last_year.value
        d = d[d['last_review'].dropna().dt.year.between(ylo, yhi, inclusive='both')]
    if w_neigh_filter.value and 'neighbourhood' in d.columns:
        key = w_neigh_filter.value.strip().lower()
        d = d[d['neighbourhood'].str.lower().str.contains(key, na=False)]
    return d

def show_kpis(df):
    total = len(df)
    price_med = df['price'].median() if 'price' in df.columns else np.nan
    rev_med = df['number_of_reviews'].median() if 'number_of_reviews' in df.columns else np.nan
    txt = f"Annonces: {total} | Prix médian: {price_med:.0f} | Avis médians: {rev_med:.0f}"
    print(txt)

def plot_price_hist(df):
    if 'price' not in df.columns or df['price'].dropna().empty:
        print("Aucune colonne prix disponible")
        return
    plt.figure(figsize=(8,4))
    plt.hist(df['price'].dropna().values, bins=w_bins.value)
    plt.title("Distribution des prix par nuit")
    plt.xlabel("Prix")
    plt.ylabel("Fréquence")
    plt.show()

def top_neighbourhoods(df, n=10):
    if 'neighbourhood' not in df.columns or 'price' not in df.columns:
        return pd.DataFrame()
    g = (df.groupby('neighbourhood', as_index=False)
           .agg(count=('id','count') if 'id' in df.columns else ('neighbourhood','count'),
                price_med=('price','median'),
                price_p25=('price',lambda s: np.nanpercentile(s,25)),
                price_p75=('price',lambda s: np.nanpercentile(s,75))))
    g = g.sort_values(['price_med','count'], ascending=[True, False]).head(n)
    return g

out = widgets.Output()

def refresh(_=None):
    with out:
        clear_output(wait=True)
        d = apply_filters(data)
        show_kpis(d)
        display(d.head(10))
        plot_price_hist(d)
        display(top_neighbourhoods(d, n=w_topn.value))

for w in [w_city, w_room, w_price, w_min_nights, w_min_reviews, w_last_year, w_outliers, w_neigh_filter, w_topn, w_bins]:
    w.observe(refresh, names='value')

display(out)
refresh()


Estimation de rendement brut simple

In [ ]:

w_buy_price = widgets.FloatText(value=200000.0, description="Prix d'achat (€)")
w_occupancy = widgets.FloatSlider(value=0.65, min=0.2, max=0.95, step=0.01, description="Taux occupation")
w_price_percentile = widgets.IntSlider(value=50, min=10, max=90, step=5, description="Pct prix")
box_rend = widgets.HBox([w_buy_price, w_occupancy, w_price_percentile])
display(box_rend)

out_yield = widgets.Output()

def compute_yield(_=None):
    with out_yield:
        clear_output(wait=True)
        d = apply_filters(data)
        if 'price' not in d.columns or d['price'].dropna().empty:
            print("Prix indisponibles")
            return
        nightly = np.nanpercentile(d['price'], w_price_percentile.value)
        annual_rev = nightly * 365 * w_occupancy.value
        gross_yield = (annual_rev / w_buy_price.value) * 100 if w_buy_price.value else np.nan
        print(f"Prix nuit (percentile {w_price_percentile.value}): {nightly:.0f} €")
        print(f"Revenus annuels estimés: {annual_rev:.0f} €")
        print(f"Rendement brut estimé: {gross_yield:.2f} %")

for w in [w_buy_price, w_occupancy, w_price_percentile, w_city, w_room, w_price, w_min_nights, w_min_reviews, w_last_year, w_outliers, w_neigh_filter]:
    w.observe(compute_yield, names='value')

display(out_yield)
compute_yield()
